usage

```python
save_scope_settings(scope, "glitch_test.json")      # trigger + glitch + LA + clock reference

# load not tested
# later:
scope = cw.scope()
scope.default_setup(verbose=False)
scope.clock.clkgen_freq = 10e6                       # set the clock first
load_scope_settings(scope, "glitch_test.json")
```

In [ ]:
import json
import time

# Properties that cannot be set (read-only or derived). Skipped when loading.
READ_ONLY = {
    "trigger": set(),
    "glitch": {"phase_shift_steps", "mmcm_locked"},
    "LA": {"present", "locked", "clkgen_enabled", "sampling_clock_frequency"},
}

# Settings that must be applied first, in this order. Everything else follows.
LOAD_ORDER = {
    "trigger": ["module", "triggers"],
    "glitch": ["enabled", "clk_src", "num_glitches", "trigger_src", "arm_timing",
               "output", "ext_offset", "repeat", "offset", "width"],
    "LA": ["enabled", "clk_source", "capture_group", "oversampling_factor",
           "downsample", "capture_depth", "trigger_source"],
}

# Order in which the sub-objects are restored (the clock must be set before these).
DEFAULT_PARTS = ("trigger", "glitch", "LA")


def settings_to_dict(obj):
    """Read all public, readable properties of a ChipWhisperer settings object."""
    if hasattr(obj, "_dict_repr"):
        try:
            return dict(obj._dict_repr())
        except Exception:
            pass
    d = {}
    for name in dir(type(obj)):
        if name.startswith("_"):
            continue
        if isinstance(getattr(type(obj), name, None), property):
            try:
                d[name] = getattr(obj, name)
            except Exception as e:
                d[name] = f"<unreadable: {e}>"
    return d


def _jsonable(v):
    if isinstance(v, (int, float, str, bool)) or v is None:
        return v
    if isinstance(v, (list, tuple)):
        return [_jsonable(x) for x in v]
    if isinstance(v, dict):
        return {str(k): _jsonable(x) for k, x in v.items()}
    return str(v)


def save_scope_settings(scope, path="scope_settings.json", parts=DEFAULT_PARTS):
    """Write trigger, glitch and LA settings (and the clock, for reference) to JSON."""
    data = {"saved": time.strftime("%Y-%m-%d %H:%M:%S")}
    for p in parts:
        obj = getattr(scope, p, None)
        if obj is not None:
            data[p] = _jsonable(settings_to_dict(obj))
    # the clock is stored for reference only, because the glitch and LA values depend on it
    data["clock_reference"] = _jsonable(settings_to_dict(scope.clock))
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return data


def load_scope_settings(scope, path="scope_settings.json", parts=DEFAULT_PARTS,
                        verbose=True):
    """Restore settings. Read-only, unreadable and unknown fields are skipped."""
    with open(path) as f:
        data = json.load(f)

    failed = []
    for p in parts:
        obj = getattr(scope, p)
        saved = data.get(p, {})
        skip = READ_ONLY.get(p, set())
        order = [k for k in LOAD_ORDER.get(p, []) if k in saved]
        order += [k for k in saved if k not in order]

        for pass_no in (1, 2):                      # second pass for dependent values
            todo = order if pass_no == 1 else [k for (q, k, _) in failed if q == p]
            if pass_no == 2:
                failed = [f for f in failed if f[0] != p]
            for k in todo:
                v = saved[k]
                if k in skip or (isinstance(v, str) and v.startswith("<unreadable")):
                    continue
                try:
                    setattr(obj, k, v)
                except Exception as e:
                    failed.append((p, k, e))

    if verbose and failed:
        for p, k, e in failed:
            print(f"skipped {p}.{k} = {data[p][k]!r}: {e}")
    return failed